In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import os
import json
import re
from util import load_json
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report


In [6]:
import re

def text_to_json(text):
    """
    Transform a formatted text string into a JSON object.
    The text should contain sections marked with EXPLANATION:, CLASSIFICATION:, and REASONING:
    
    Args:
        text (str): The input text to transform
        
    Returns:
        dict: A dictionary containing the parsed sections
    """
    # Regex to match each section
    explanation_match = re.search(r'EXPLANATION:\s*(.*?)(?=(CLASSIFICATION:|REASONING:|$))', text, re.S)
    classification_match = re.search(r'CLASSIFICATION:\s*(.*?)(?=(EXPLANATION:|REASONING:|$))', text, re.S)
    reasoning_match = re.search(r'REASONING:\s*(.*?)(?=(EXPLANATION:|CLASSIFICATION:|$))', text, re.S)
    
    # Extract the content for each section if it exists
    result = {
        'explanation': explanation_match.group(1).strip() if explanation_match else '',
        'classification': classification_match.group(1).strip() if classification_match else '',
        'reasoning': reasoning_match.group(1).strip() if reasoning_match else '',
    }
    
    return result


In [19]:
def category_to_binary_label(category):
    if "HIGH" in category:
        #print('0 == high --> ',category)
        return 0
    elif "MEDIUM" in category:
        #print('1 == MEDIUM --> ',category)
        return 1
    elif "LOW" in category:
        print('2 == LOW --> ',category)
        return 2
    else:
        raise ValueError(f"Invalid category value: {category}")

def transalte_importance_score(file_name):
    if "i_1" in file_name:
        return 0
    elif "i_2" in file_name:
        return 0
    elif "i_3" in file_name:
        return 1
    elif "i_4" in file_name:
        return 2
    else:
        raise ValueError(f"Invalid file name: {file_name}")


def get_df(output_dir = './results_facts_law_3c/'):
    results = os.listdir(output_dir)

    data = []
    for result in results:
        result_data = load_json(os.path.join(output_dir, result))
        syth_output = text_to_json(result_data["synthetizer_response"])
        ground_truth = transalte_importance_score(result)
        data.append(
            {
                "file_name": result,
                "classification": category_to_binary_label(syth_output["classification"]),
                "ground_truth": ground_truth,
            }
        )

    df = pd.DataFrame(data)
    return df

def get_heatmap(df: pd.DataFrame, target_col: str, pred_col: str):
    """Plot and display a heatmap of the confusion matrix."""
    import seaborn as sns
    import matplotlib.pyplot as plt

    cm = confusion_matrix(df[target_col], df[pred_col], normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()


def get_metrics_binary(df):
    # Key Case vs. All
    # Ground truth: 1 for key case, 0 for not key case
    y_true = df['ground_truth']
    # Model predictions: 1 for key case, 0 for not key case
    y_pred = df['classification']

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")

    # Classification report for additional insights
    rreport = classification_report(y_true, y_pred, target_names=["HIGH", "MEDIUM", "LOW"], output_dict=True)
    report = classification_report(y_true, y_pred, target_names=["HIGH", "MEDIUM", "LOW"])

    # Extracting micro and macro F1 scores
    micro_f1 = rreport["accuracy"]  # Micro F1 is equivalent to accuracy in binary classification
    macro_f1 = rreport["macro avg"]["f1-score"]

    # Presenting results
    metrics = {

        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Micro F1": micro_f1,
        "Macro F1": macro_f1,
    }
    return metrics, report


In [20]:
output_dir = './results_facts_law_3c/'
df = get_df(output_dir)
metrics, report = get_metrics_binary(df)
metrics

2 == LOW -->  LOW
2 == LOW -->  LOW


{'Accuracy': 0.5913705583756346,
 'Precision': 0.6022222222222222,
 'Recall': 0.3377608680335376,
 'F1 Score': 0.30900868219757366,
 'Micro F1': 0.5913705583756346,
 'Macro F1': 0.30900868219757366}

In [21]:
print(report)

              precision    recall  f1-score   support

        HIGH       0.64      0.90      0.75       249
      MEDIUM       0.17      0.07      0.10        95
         LOW       1.00      0.04      0.08        50

    accuracy                           0.59       394
   macro avg       0.60      0.34      0.31       394
weighted avg       0.57      0.59      0.51       394

